<a href="https://colab.research.google.com/github/NicolasDonizete/Processamento-de-Dados-Massivos-MapReduce/blob/main/Processamento_de_Dados_Massivos_MapReduce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
from functools import reduce
from datetime import datetime
import pandas as pd


PAYMENT_TYPES = {
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip",
}


url = "https://huggingface.co/datasets/alexvaroz/nyc_taxi_trip_2024_p1_sample/resolve/main/nyc_tripdata_2024_sample_1M.csv"


df = pd.read_csv(url)


data = df.to_dict(orient="records")
print(f"Total de registros carregados: {len(data)}")

Total de registros carregados: 1000000


In [16]:
def mapper_group_by_key(mapped_records):

    grouped = {}
    for key, value in mapped_records:
        if key not in grouped:
            grouped[key] = []
        grouped[key].append(value)
    return grouped


def map_reduce(data_list, map_func, reduce_func):

    mapped = list(map(map_func, data_list))

    mapped = [item for item in mapped if item is not None]


    grouped = mapper_group_by_key(mapped)


    results = {}
    for key, values in grouped.items():
        results[key] = reduce(reduce_func, values)

    return results

In [17]:

def map_trips_by_payment(record):
    pt = record.get("payment_type")
    if pd.notna(pt):
        payment_name = PAYMENT_TYPES.get(int(pt), "Unknown/Other")
        return (payment_name, 1)
    return None

def reduce_count(acc, val):
    return acc + val

q1_result = map_reduce(data, map_trips_by_payment, reduce_count)

print("--- 1. Número de viagens por tipo de pagamento ---")
for payment, count in q1_result.items():
    print(f"{payment}: {count:,} viagens")

--- 1. Número de viagens por tipo de pagamento ---
Cash: 136,221 viagens
Credit card: 743,405 viagens
Unknown/Other: 97,124 viagens
Dispute: 16,543 viagens
No charge: 6,707 viagens


In [6]:

def map_revenue_by_payment(record):
    pt = record.get("payment_type")
    total = record.get("total_amount")
    if pd.notna(pt) and pd.notna(total):
        payment_name = PAYMENT_TYPES.get(int(pt), "Unknown/Other")
        return (payment_name, float(total))
    return None



def reduce_sum(acc, val):
    return acc + val


q2_result = map_reduce(data, map_revenue_by_payment, reduce_sum)

print("--- 2. Receita total por tipo de pagamento ---")
for payment, total in q2_result.items():
    print(f"{payment}: ${total:,.2f}")

--- 2. Receita total por tipo de pagamento ---
Cash: $3,168,095.90
Credit card: $21,785,219.95
Unknown/Other: $2,376,069.77
Dispute: $25,214.51
No charge: $53,932.48


In [8]:

def map_fare(record):
    fare = record.get("fare_amount")
    if pd.notna(fare):
        return ("fare", (float(fare), 1))
    return None



def reduce_fare_stats(acc, val):
    return (acc[0] + val[0], acc[1] + val[1])


q3_intermediate = map_reduce(data, map_fare, reduce_fare_stats)


total_fare, count_fare = q3_intermediate["fare"]
average_fare = total_fare / count_fare

print("--- 3. Tarifa média cobrada nas viagens ---")
print(f"Tarifa média: ${average_fare:.2f}")

--- 3. Tarifa média cobrada nas viagens ---
Tarifa média: $18.86


In [9]:

def map_longest_trip(record):
    dist = record.get("trip_distance")
    pickup = record.get("tpep_pickup_datetime")
    if pd.notna(dist) and pd.notna(pickup):
        return ("longest_trip", (float(dist), str(pickup)))
    return None



def reduce_max_distance(acc, val):
    return acc if acc[0] >= val[0] else val



q4_result = map_reduce(data, map_longest_trip, reduce_max_distance)
max_dist, pickup_time = q4_result["longest_trip"]

print("--- 4. Viagem mais longa ---")
print(f"Data e Hora: {pickup_time}")
print(f"Distância: {max_dist} milhas")

--- 4. Viagem mais longa ---
Data e Hora: 2024-05-10 17:33:00
Distância: 86789.2 milhas


In [10]:

def map_trips_by_hour(record):
    pickup = record.get("tpep_pickup_datetime")
    if pd.notna(pickup):
        try:

            dt = datetime.strptime(str(pickup), "%Y-%m-%d %H:%M:%S")
            return (dt.hour, 1)
        except ValueError:
            return None
    return None



q5_result = map_reduce(data, map_trips_by_hour, reduce_count)

print("--- 5. Quantidade de viagens por hora ---")
for hour in sorted(q5_result.keys()):
    print(f"Hora {hour:02d}h: {q5_result[hour]:,} viagens")

--- 5. Quantidade de viagens por hora ---
Hora 00h: 29,165 viagens
Hora 01h: 18,822 viagens
Hora 02h: 12,280 viagens
Hora 03h: 8,281 viagens
Hora 04h: 6,054 viagens
Hora 05h: 6,194 viagens
Hora 06h: 13,966 viagens
Hora 07h: 28,065 viagens
Hora 08h: 38,308 viagens
Hora 09h: 42,309 viagens
Hora 10h: 44,804 viagens
Hora 11h: 48,295 viagens
Hora 12h: 53,128 viagens
Hora 13h: 55,362 viagens
Hora 14h: 59,345 viagens
Hora 15h: 60,205 viagens
Hora 16h: 61,563 viagens
Hora 17h: 67,880 viagens
Hora 18h: 71,403 viagens
Hora 19h: 62,752 viagens
Hora 20h: 56,542 viagens
Hora 21h: 58,333 viagens
Hora 22h: 54,581 viagens
Hora 23h: 42,363 viagens


In [11]:

def map_distance_by_hour(record):
    pickup = record.get("tpep_pickup_datetime")
    dist = record.get("trip_distance")
    if pd.notna(pickup) and pd.notna(dist):
        try:
            dt = datetime.strptime(str(pickup), "%Y-%m-%d %H:%M:%S")
            return (dt.hour, float(dist))
        except ValueError:
            return None
    return None



q6_result = map_reduce(data, map_distance_by_hour, reduce_sum)

print("--- 6. Distância total percorrida por hora ---")
for hour in sorted(q6_result.keys()):
    print(f"Hora {hour:02d}h: {q6_result[hour]:,.2f} milhas")

--- 6. Distância total percorrida por hora ---
Hora 00h: 109,568.73 milhas
Hora 01h: 60,695.19 milhas
Hora 02h: 36,643.43 milhas
Hora 03h: 29,745.90 milhas
Hora 04h: 28,350.30 milhas
Hora 05h: 91,136.53 milhas
Hora 06h: 131,056.86 milhas
Hora 07h: 195,237.07 milhas
Hora 08h: 169,111.73 milhas
Hora 09h: 222,652.02 milhas
Hora 10h: 138,342.76 milhas
Hora 11h: 145,189.30 milhas
Hora 12h: 166,186.16 milhas
Hora 13h: 190,847.83 milhas
Hora 14h: 215,732.27 milhas
Hora 15h: 312,374.38 milhas
Hora 16h: 238,718.29 milhas
Hora 17h: 321,659.21 milhas
Hora 18h: 252,601.19 milhas
Hora 19h: 265,021.98 milhas
Hora 20h: 267,461.66 milhas
Hora 21h: 283,775.08 milhas
Hora 22h: 189,960.49 milhas
Hora 23h: 161,753.45 milhas
